# Введение в LangChain. Модель, промпт, агент

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* https://openrouter.ai/
* https://openrouter.ai/mistralai/mistral-7b-instruct:free
* https://www.aurelio.ai/learn/langchain-lcel
* https://www.youtube.com/watch?v=jjJcBdvBgSY&t=170s
* https://docs.langchain.com/oss/python/migrate/langchain-v1

* https://docs.langchain.com/oss/python/langchain/models
* https://docs.langchain.com/oss/python/langchain/messages
* https://docs.langchain.com/oss/python/langchain/agents
* https://docs.langchain.com/oss/python/langchain/tools
* https://docs.langchain.com/oss/python/langchain/structured-output
* https://docs.langchain.com/oss/python/langchain/streaming/overview


## Задачи для совместного разбора

1. Обсудите основные концепты фреймворка LangChain для построения приложений с использованием LLM: промпты (и их виды) и runnables.

In [3]:
# pip install openai

In [4]:
from openai import OpenAI

In [7]:
# client = OpenAI(
#     base_url="https://ollama.findatalab.ru/v1",
#     api_key="ollama"
# )

In [1]:
# client.chat.completions.create(
#     model="fin/GPT:latest",
#     messages=[
#         {"role": "user", "content": "Объясни что такое аркадиум"}
#     ]
# )

In [11]:
import langchain

langchain.__version__

'1.2.15'

In [ ]:
# import os
# from langchain.chat_models import init_chat_model
# from google.colab import userdata


# os.environ["OPENAI_API_BASE"] = "https://api.proxyapi.ru/openai/v1"
# os.environ["OPENAI_API_KEY"] = userdata.get('PROXYAPI_KEY')
# model = init_chat_model(
#     model="gpt-4o-mini"
# )

In [ ]:
# model.invoke(["Объясни что такое аркадиум"])

AIMessage(content='Аркадиум — это условное название для концепции или места, которое ассоциируется с идеалов пасторальной жизни, красоты природы и гармонии. Чаще всего этот термин используется в литературе, искусстве и философии для обозначения утопического пространства, где люди живут в согласии с природой и друг с другом.\n\nВ контексте разных культурных и исторических эпох аркадиум может иметь различные интерпретации. В античной Греции Аркадия была известна как область, символизирующая простую и идиллическую жизнь вдали от городской суеты.\n\nКроме того, "аркадиум" может использоваться в различных художественных произведениях для обозначения идеального социального или культурного пространства, где царят мир и счастье.\n\nЕсли вы имеете в виду что-то конкретное, например, определенный контекст или использование этого слова, пожалуйста, уточните, и я постараюсь помочь более подробно!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 205, 'p

In [ ]:
# from pydantic import BaseModel, Field

# class Decision(BaseModel):
#   decision: str = Field(description="Решение о том уйти ли мне со следующей пары")
#   reason: str = Field(description="Аргументация для препода")

# struct_model = model.with_structured_output(Decision)

In [ ]:
# struct_model.invoke(["Я посмотрел сегодня слишком много тиктоков. Уйти ли мне домой?"])

Decision(decision='Да, стоит уйти домой, чтобы отдохнуть и восстановить силы.', reason='Слишком много времени, проведенного за просмотром тиктоков, может быть утомительным. Лучше сменить обстановку, отдохнуть и отвлечься от экрана.')

In [ ]:
# from math import isqrt
# from langchain_core.tools import tool

# @tool
# def isqrt_tool(x: int) -> int:
#   """Вычисляет isqrt от натурального числа"""
#   return isqrt(x)

In [ ]:
# isqrt_tool

StructuredTool(name='isqrt_tool', description='Вычисляет isqrt от вещественного числа', args_schema=<class 'langchain_core.utils.pydantic.isqrt_tool'>, func=<function isqrt_tool at 0x7e161467ff60>)

In [ ]:
# from langchain_openai import ChatOpenAI

# model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools([isqrt_tool])

In [ ]:
# resp = model.invoke("Извлеки isqrt из числа 143")

In [ ]:
# resp.content

''

In [ ]:
# resp.tool_calls

[{'name': 'isqrt_tool',
  'args': {'x': 143},
  'id': 'call_6P3iyRZM7YyRNQV7IIB0VFyG',
  'type': 'tool_call'}]

In [ ]:
# # from langchain.agents import create_agent

# agent = create_agent(model="gpt-4o-mini", tools=[isqrt_tool])

In [54]:
import os
import datetime
import time
import numpy as np
from typing import List, Literal, Dict, Any
from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool, ToolException
from langchain_core.tools.base import InjectedToolCallId
from langchain_core.tools import ToolException
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1" 

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.0
)

## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Используя пакет `langchain`, реализуйте цикл взаимодействия с любой доступной большой языковой моделью. Ваша цель - задавая вопросы и предоставляя информацию, чтобы оценить, насколько хорошо модель удерживает контекст. Для этого вы должны накапливать набор сообщений (ваших и ответов модели) и каждый раз передавать весь диалог на вход. Для передачи сообщений организуйте ввод через функцию `input`. Диалог заканчивается, когда введено слово "exit".


В качестве первого сообщения диалога укажите следующий системный промпт:

"Ты — эксперт в области обработки естественного языка. В ходе нашего диалога ты должен запоминать информацию и использовать ее в ответах на мои вопросы. Тема нашего обсуждения — NLP, включая методы обработки текста и алгоритмы машинного обучения."

Задайте несколько общих вопросов. Последним вопросом уточните что-то, что модель ответила ранее. Сделайте выводы.


- [ ] Проверено на семинаре

In [9]:
def manual_chat_loop(model: ChatOpenAI):
    chat_history: List[Any] =[
        SystemMessage(
            content="Ты — эксперт в области обработки естественного языка. В ходе нашего диалога ты должен запоминать информацию и использовать ее в ответах на мои вопросы. Тема нашего обсуждения — NLP, включая методы обработки текста и алгоритмы машинного обучения."
        )
    ]
    
    display(Markdown("`model`: Привет! Я эксперт по NLP. Задавай вопросы! (Для выхода введи 'exit')\n"))
    
    while True:
        user_input = input("user : ")
        display(Markdown(f"`user`: {user_input}\n"))
        if user_input.strip().lower() == "exit":
            display(Markdown("`model`: До встречи!"))
            break
            
        chat_history.append(HumanMessage(content=user_input))
        response = model.invoke(chat_history)
        display(Markdown(f"`model`: {response.content}\n"))
        chat_history.append(AIMessage(content=response.content))

manual_chat_loop(llm)

`model`: Привет! Я эксперт по NLP. Задавай вопросы! (Для выхода введи 'exit')


`user`: Обясняй мне всё как пятилетнему ребенку, который любит варить суп и трансформеров. Что такое эмбеддинги?


`model`: Представь, что у тебя есть много разных игрушек — трансформеров. Каждый трансформер может быть разного цвета, размера и формы. Теперь представь, что ты хочешь, чтобы все твои игрушки были в одном месте, и ты мог легко их находить.

Эмбеддинги — это как волшебные карточки, которые помогают нам понять, как похожи или отличаются разные слова. Каждое слово, как твой трансформер, получает свою карточку с числами. Эти числа показывают, насколько слово похоже на другие слова. Например, слова "кот" и "собака" будут иметь похожие карточки, потому что это оба животных, а слово "стол" будет иметь совсем другую карточку, потому что это не животное.

Так что эмбеддинги помогают компьютерам понимать слова и находить их связи, как ты находишь своих трансформеров по цвету или размеру!


`user`: Какие ты знаешь модели, которые вычисляют вектора-эмбеддинги?


`model`: Есть несколько популярных моделей, которые помогают вычислять вектора-эмбеддинги для слов. Вот некоторые из них:

1. **Word2Vec**: Эта модель учит слова представлять в виде векторов, основываясь на том, как часто они встречаются рядом друг с другом в тексте. Она может использовать два подхода: Skip-gram и Continuous Bag of Words (CBOW).

2. **GloVe (Global Vectors for Word Representation)**: Эта модель также создает векторы для слов, но делает это, анализируя глобальные статистики слов в большом корпусе текста. Она пытается понять, как часто слова появляются вместе и строит векторы на основе этих частот.

3. **FastText**: Эта модель, разработанная Facebook, улучшает Word2Vec, учитывая не только целые слова, но и их части (н-граммы). Это помогает лучше справляться с редкими словами и морфологией.

4. **BERT (Bidirectional Encoder Representations from Transformers)**: Эта модель использует трансформеры и может создавать контекстуальные эмбеддинги, что означает, что одно и то же слово может иметь разные векторы в зависимости от контекста, в котором оно используется.

5. **Sentence-BERT**: Это расширение BERT, которое позволяет получать векторы для целых предложений, а не только для отдельных слов. Это полезно для задач, связанных с пониманием смыслов предложений.

Эти модели помогают компьютерам лучше понимать текст и находить связи между словами!


`user`: Я ничего не понял...


`model`: Давай объясню проще, как будто ты снова маленький повар, который варит суп!

Представь, что у тебя есть разные ингредиенты для супа: морковь, картошка, лук и т.д. Каждый ингредиент — это как слово. Чтобы понять, как они могут сочетаться в супе, нам нужно знать, как они похожи друг на друга.

1. **Word2Vec**: Это как если бы ты смотрел, какие ингредиенты часто используются вместе. Если морковь и картошка часто идут в одном супе, значит, они похожи. Word2Vec помогает компьютеру понять, какие слова часто встречаются рядом.

2. **GloVe**: Это как если бы ты смотрел на все свои рецепты и считал, сколько раз каждый ингредиент используется. Если морковь и картошка часто встречаются в разных рецептах, значит, они тоже похожи. GloVe делает что-то похожее, но с большим количеством текста.

3. **FastText**: Это как если бы ты разбивал ингредиенты на маленькие кусочки. Например, "морковь" можно разбить на "мор" и "ковь". Это помогает понять, как разные слова связаны, даже если они не очень популярны.

4. **BERT**: Это как если бы ты готовил суп и понимал, что морковь может быть сладкой, если ты добавил её в начале, и не такой сладкой, если добавил в конце. BERT помогает компьютеру понимать, как слова меняются в зависимости от того, с чем они "смешиваются".

5. **Sentence-BERT**: Это как если бы ты готовил целый суп и хотел понять, как все ингредиенты работают вместе. Sentence-BERT помогает компьютеру понимать целые предложения, а не только отдельные слова.

Надеюсь, теперь стало понятнее! Эмбеддинги — это как волшебные карточки для слов, которые помогают компьютерам понимать, как они связаны друг с другом, как ты понимаешь, как разные ингредиенты работают в супе!


`user`: exit


`model`: До встречи!

модель запомнила, что я люблю суп. 

<p class="task" id="2"></p>

2\. Вам дано два фрагмента текста. В одном из говорится, почему автор любит котиков, а в другом - почему автор любит собачек.

Создайте конвейер, который заставит модель вывести (в зависимости от входного сообщения) ответ в следующем виде:
```json
{
    "animal": "кошка",
    "reason": "..."
}
```

или

```json
{
    "animal": "собака",
    "reason": "..."
}
```
Для этого опишите желаемую структуру ответа с помощью `pydantic` и создайте модель с поддержкой структуры. Продемонстрируйте работу на примерах.


- [ ] Проверено на семинаре

In [11]:
cats_lover_msg = "Я обожаю кошек за их независимый характер и мягкие шершащие лапки, которые всегда радуют меня своим прикосновением."
dogs_lover_msg = "Собак я люблю за их преданность и энергичность, которые делают каждую прогулку незабываемой и полной веселья."

In [10]:
class AnimalReason(BaseModel):
    """Схема для извлечения информации о животном и причине его любви."""
    animal: Literal["кошка", "собака"] = Field(
        description="Вид животного, упомянутый в тексте. Строго 'кошка' или 'собака'."
    )
    reason: str = Field(
        description="Точная причина, почему автор любит это животное."
    )

In [12]:
structured_llm = llm.with_structured_output(AnimalReason)

In [13]:
cat_result = structured_llm.invoke([HumanMessage(content=cats_lover_msg)])
print(f"Тип объекта: {type(cat_result)}")
print(cat_result.model_dump_json(indent=2, ensure_ascii=False))

Тип объекта: <class '__main__.AnimalReason'>
{
  "animal": "кошка",
  "reason": "Я обожаю кошек за их независимый характер и мягкие шершащие лапки, которые всегда радуют меня своим прикосновением."
}


In [14]:
dog_result = structured_llm.invoke([HumanMessage(content=dogs_lover_msg)])
print(dog_result.model_dump_json(indent=2, ensure_ascii=False))

{
  "animal": "собака",
  "reason": "Собак я люблю за их преданность и энергичность, которые делают каждую прогулку незабываемой и полной веселья."
}


In [15]:
dogs_lover_idk_msg = "Я не могу определиться, люблю ли я собак или кошек больше. Они оба такие классные!"

dog_result = structured_llm.invoke([HumanMessage(content=dogs_lover_idk_msg)])
print(dog_result.model_dump_json(indent=2, ensure_ascii=False))

{
  "animal": "собака",
  "reason": "Собаки очень преданные и всегда рады видеть своего хозяина. Они умеют поддерживать и поднимать настроение, а также становятся отличными компаньонами для активного отдыха."
}


Круто придумал...

<p class="task" id="3"></p>

3\. Напишите обычную функцию `get_joke(topic: str)` и задекорируйте при помощи `@tool`. Добавьте ей docstring. Опишите аргументы таким образом, чтобы иметь возможность искать анекдот в заданном словаре.

Создайте модель с привязанным инструментом. Попросите модель рассказать шутку на выбранную тему. Проанализируйте ответ. Убедитесь, что модель не стала генерировать текст сама. Выведите `response.tool_calls`. Убедитесь, что там правильно определились аргументы.

Вручную вызовите предложенный инструмент с аргументами, которые предлагает модель. Прокомментируйте результат.

Если ваша модель не поддерживает работу с инструментами, рекомендуется выбрать другую модель. Если такой возможности нет, сэмулируйте работу с инструментами при помощи structured output.

- [ ] Проверено на семинаре

In [26]:
JOKES = {
    "программисты": [
        "Программист — это человек, который решает проблему, о существовании которой вы не знали, способом, который вы не понимаете.",
        "Заходит программист в бар, а бармен ему говорит: 'Почему ты такой грустный?' Программист отвечает: 'У меня баги в коде, и я не могу их найти.' Бармен советует: 'Попробуй перезагрузиться!'"
    ],
    "искусственный интеллект": [
        "Искусственный интеллект — это когда компьютер делает вид, что умнее человека, а человек делает вид, что так и задумано.",
        "Задаешь ИИ вопрос, а он отвечает: 'You're absolutely right!'"
    ],
    "математика": [
        "Математики шутят редко, но если шутят — доказательство прилагается.",
        "Один математик говорит другому: 'Ты знаешь, я придумал новую шутку про бесконечность.' Другой отвечает: 'Расскажи!' Первый говорит: 'Ну, она бесконечно смешная!'"
    ],
}


In [27]:
JOKES['математика']

['Математики шутят редко, но если шутят — доказательство прилагается.',
 "Один математик говорит другому: 'Ты знаешь, я придумал новую шутку про бесконечность.' Другой отвечает: 'Расскажи!' Первый говорит: 'Ну, она бесконечно смешная!'"]

In [28]:
@tool
def get_joke(topic: str) -> str:
    """
    Ищет и возвращает анекдот на заданную тему из внутренней базы.
    Используйте этот инструмент, когда пользователь просит пошутить на конкретную тему.
    """
    topic = topic.lower()
    if topic in JOKES.keys():
        answer = JOKES[topic][np.random.randint(len(JOKES[topic]))]
        return answer
    return "Извините, шутки на эту тему не найдено."   

llm_with_tools = llm.bind_tools([get_joke])

query = "Расскажи анекдот про математику, пожалуйста!"
response = llm_with_tools.invoke([HumanMessage(content=query)])
print("Ответ модели:", repr(response.content))
print("Запросы к инструментам (tool_calls):\n", response.tool_calls)

Ответ модели: ''
Запросы к инструментам (tool_calls):
 [{'name': 'get_joke', 'args': {'topic': 'математика'}, 'id': 'call_vFroBUXrqZqoMRdvmQ0KaHWa', 'type': 'tool_call'}]


In [29]:
if response.tool_calls:
    tool_call = response.tool_calls[0]
    function_name = tool_call["name"]
    arguments = tool_call["args"]
    
    print(f"\nМодель просит вызвать функцию '{function_name}' с аргументами {arguments}")
    
    if function_name == "get_joke":
        result = get_joke.invoke(arguments) 
        print(f"\nРезультат выполнения функции: {result}")


Модель просит вызвать функцию 'get_joke' с аргументами {'topic': 'математика'}

Результат выполнения функции: Один математик говорит другому: 'Ты знаешь, я придумал новую шутку про бесконечность.' Другой отвечает: 'Расскажи!' Первый говорит: 'Ну, она бесконечно смешная!'


<p class="task" id="4"></p>

4\. Используя ранее созданную функцию `get_joke` и ту же модель, создайте агента, который умеет использовать инструмент `get_joke`. Агент не должен генерировать анекдоты самостоятельно — он обязан получать их только через инструмент.
Попросите агента рассказать шутку на заданную тему. При помощи системного промпта уточните, что если в "базе" нет нужной темы, нужно вывести сообщение, содержащее список доступных тем. Для получения этих списков реализуйте отдельный tool. Добавьте tool error handling для `get_joke`, чтобы подтолкнуть LLM вызвать инструмент для получения списка тем в случае ошибки.

Проанализируйте работу агента на нескольких примерах (примеры успешного поиска и неуспешного поиска). Убедитесь, что агент вызвал инструмент, выведите все `tool_calls`, которые сделал агент, проверьте корректность аргументов.

Если ваша модель не поддерживает работу с инструментами, рекомендуется выбрать другую модель. Если такой возможности нет, сэмулируйте работу с инструментами при помощи structured output.

- [ ] Проверено на семинаре


In [46]:
@tool
def get_available_topics() -> str:
    """Возвращает список всех доступных тем для анекдотов."""
    return f"В базе есть шутки на следующие темы: {', '.join(JOKES.keys())}"

@tool
def get_joke_robust(topic: str) -> str:
    """Ищет и возвращает анекдот на заданную тему из внутренней базы."""
    topic = topic.lower()
    if topic in JOKES.keys():
        import numpy as np
        return JOKES[topic][np.random.randint(len(JOKES[topic]))]
    else:
        return (f"ОШИБКА: Темы '{topic}' нет в базе. Срочно вызови инструмент get_available_topics, чтобы узнать список тем.")

In [ ]:
tools = [get_joke_robust, get_available_topics]
llm_with_tools = llm.bind_tools(tools)

In [51]:

def chatbot_node(state: MessagesState):
    """Узел, который отвечает за вызов языковой модели."""
    sys_prompt = SystemMessage(
        content="""Ты — веселая ИИ-комикесса. Твоя задача — рассказывать анекдоты по запросу пользователя.
                ПРАВИЛА:
                0. Перед тем, как ответить на запрос пользователя о шутке, обязательно суммаризируй тему до максимально простого одного слова (например, "шутка про программистов" -> "программисты").
                1. НЕ придумывай анекдоты сам. Бери их ТОЛЬКО из инструмента get_joke_robust.
                2. Если инструмент выдал ошибку, ОБЯЗАТЕЛЬНО вызови get_available_topics.
                3. Получив список тем, извинись перед пользователем и предложи ему выбрать из доступного списка."""
    )
    messages = [sys_prompt] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

graph_builder = StateGraph(MessagesState)

graph_builder.add_node("chatbot", chatbot_node)
graph_builder.add_node("tools", ToolNode(tools=tools)) 
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
joke_agent_app = graph_builder.compile()

def run_manual_graph(query: str):
    display(Markdown(f"`user`Запрос пользователя: {query}"))
    result = joke_agent_app.invoke({"messages": [HumanMessage(content=query)]})
    
    for msg in result["messages"]:
        if msg.type == "human":
            continue
        elif msg.type == "ai":
            if msg.tool_calls:
                display(Markdown(f"`model`: Вызываю инструмент '{msg.tool_calls[0]['name']}'"))
            if msg.content:
                display(Markdown(f"`model`: {msg.content}"))
        elif msg.type == "tool":
            display(Markdown(f"`tool`: {msg.content}"))


In [52]:
run_manual_graph("Расскажи шутку про программистов.")

`user`Запрос пользователя: Расскажи шутку про программистов.

`model`: Вызываю инструмент 'get_joke_robust'

`tool`: Программист — это человек, который решает проблему, о существовании которой вы не знали, способом, который вы не понимаете.

`model`: Программисты! Вот шутка: "Программист — это человек, который решает проблему, о существовании которой вы не знали, способом, который вы не понимаете." 😄

In [53]:
run_manual_graph("Расскажи шутку про гумманитариев.")

`user`Запрос пользователя: Расскажи шутку про гумманитариев.

`model`: Вызываю инструмент 'get_joke_robust'

`tool`: ОШИБКА: Темы 'гуманитарии' нет в базе. Срочно вызови инструмент get_available_topics, чтобы узнать список тем.

`model`: Вызываю инструмент 'get_available_topics'

`tool`: В базе есть шутки на следующие темы: программисты, искусственный интеллект, математика

`model`: Извини, но шуток про гуманитариев у меня нет. Могу предложить выбрать из доступных тем: программисты, искусственный интеллект или математика. Какую тему выберешь?

<p class="task" id="5"></p>

5\. Используя возможности агента по потоковой генерации сообщений (streaming), выведите на экран в режиме реального времени процесс получения анекдотов на 5 разных тем (существующих и не существующих в базе). Каждое сообщение выводите на экране в виде: `[Дата Время] Тип Контент`, где тип - это `model` или `tools`.

- [ ] Проверено на семинаре

In [ ]:
def stream_langgraph_updates(topics: list[str]):
    query = f"Расскажи мне по одной шутке на каждую из этих тем: {', '.join(topics)}. Если темы нет, так и скажи."
    
    display(Markdown(f"**Запрос**: {query}\n"))
    
    for chunk in joke_agent_app.stream({"messages": [HumanMessage(content=query)]}, stream_mode="updates"):
        current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        for node_name, updates in chunk.items():
            # Достаем последнее сообщение, которое вернул данный узел
            msg = updates["messages"][-1]
            
            if node_name == "chatbot":
                if msg.tool_calls:
                    display(Markdown(f"`model`: Принято решение вызвать инструмент '{msg.tool_calls[0]['name']}'"))
                if msg.content:
                    display(Markdown(f"`model`: {msg.content}"))
            
            elif node_name == "tools":
                display(Markdown(f"`tool`: {msg.content}"))
                
        time.sleep(0.5)

test_topics =["искусственный интеллект", "биология", "программисты", "физика", "математика", "программисты"]

stream_langgraph_updates(test_topics)

**Запрос**: Расскажи мне по одной шутке на каждую из этих тем: искусственный интеллект, биология, программисты, физика, математика. Если темы нет, так и скажи.


`model`: Принято решение вызвать инструмент 'get_joke_robust'

`tool`: Один математик говорит другому: 'Ты знаешь, я придумал новую шутку про бесконечность.' Другой отвечает: 'Расскажи!' Первый говорит: 'Ну, она бесконечно смешная!'

`model`: Принято решение вызвать инструмент 'get_available_topics'

`tool`: В базе есть шутки на следующие темы: программисты, искусственный интеллект, математика

`model`: Вот что у меня получилось:

- **Искусственный интеллект**: Искусственный интеллект — это когда компьютер делает вид, что умнее человека, а человек делает вид, что так и задумано.
- **Программисты**: Программист — это человек, который решает проблему, о существовании которой вы не знали, способом, который вы не понимаете.
- **Математика**: Один математик говорит другому: 'Ты знаешь, я придумал новую шутку про бесконечность.' Другой отвечает: 'Расскажи!' Первый говорит: 'Ну, она бесконечно смешная!'

К сожалению, шуток на темы "биология" и "физика" у меня нет. Если хочешь, можешь выбрать другую тему из доступного списка: программисты, искусственный интеллект, математика.

In [56]:
test_topics =["искусственный интеллект", "биология", "программисты", "физика", "математика", "программисты"]
# а если дать ему повторяющуюся тему, он должен понять, что она уже была и не вызвать инструмент второй раз, надеюсь
stream_langgraph_updates(test_topics)

**Запрос**: Расскажи мне по одной шутке на каждую из этих тем: искусственный интеллект, биология, программисты, физика, математика, программисты. Если темы нет, так и скажи.


`model`: Принято решение вызвать инструмент 'get_joke_robust'

`tool`: Заходит программист в бар, а бармен ему говорит: 'Почему ты такой грустный?' Программист отвечает: 'У меня баги в коде, и я не могу их найти.' Бармен советует: 'Попробуй перезагрузиться!'

`model`: Принято решение вызвать инструмент 'get_available_topics'

`tool`: В базе есть шутки на следующие темы: программисты, искусственный интеллект, математика

`model`: Вот что у меня получилось:

- **Искусственный интеллект**: Искусственный интеллект — это когда компьютер делает вид, что умнее человека, а человек делает вид, что так и задумано.
- **Биология**: Извините, но темы "биология" нет в базе.
- **Программисты**: Программист — это человек, который решает проблему, о существовании которой вы не знали, способом, который вы не понимаете.
- **Физика**: Извините, но темы "физика" нет в базе.
- **Математика**: Один математик говорит другому: 'Ты знаешь, я придумал новую шутку про бесконечность.' Другой отвечает: 'Расскажи!' Первый говорит: 'Ну, она бесконечно смешная!'
- **Программисты**: Заходит программист в бар, а бармен ему говорит: 'Почему ты такой грустный?' Программист отвечает: 'У меня баги в коде, и я не могу их найти.' Бармен советует: 'Попробуй перезагрузиться!'

Если хотите, можете выбрать другую тему из доступного списка: программисты, искусственный интеллект, математика!

<p class="task" id="6"></p>

6\. Повторите решение задачи 1, теперь используя агентов и их возможности по хранению памяти. Создайте агента и настройте  системный промпт. Организуйте цикл диалога с пользователем через input. Диалог должен завершаться, когда введено слово "exit". Агент должен автоматически запоминать все вопросы и ответы с помощью встроенной памяти, без ручного накопления сообщений. Задайте несколько вопросов по теме NLP, включая общие вопросы про методы и алгоритмы и вопросы, уточняющие детали, которые агент уже давал ранее.

Настройте агента так, чтобы после 4 сообщений от производил суммаризацию диалога. Продемонстрируйте работу этого механизма.

- [ ] Проверено на семинаре

In [58]:
from typing import Annotated, TypedDict, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import RemoveMessage
from langgraph.checkpoint.memory import MemorySaver

class MemoryState(TypedDict):
    messages: Annotated[list, add_messages]
    summary: str

def chat_node(state: MemoryState):
    summary = state.get("summary", "")
    sys_msg = "Ты — эксперт в области обработки естественного языка. В ходе нашего диалога ты должен запоминать информацию и использовать ее в ответах на мои вопросы. Тема нашего обсуждения — NLP, включая методы обработки текста и алгоритмы машинного обучения."
    if summary:
        sys_msg += f"\n\nСаммари прошлых бесед: {summary}"
        
    messages_to_pass = [SystemMessage(content=sys_msg)] + state["messages"]
    response = llm.invoke(messages_to_pass)
    return {"messages": [response]}

def summarizer_node(state: MemoryState):
    summary = state.get("summary", "")
    messages = state["messages"]
    
    print("\n[SYSTEM] Вызван узел суммаризации. Сжимаем контекст...")
    
    summary_prompt = (
        f"Текущее саммари: {summary}\n\n"
        f"Новые сообщения:\n{[m.content for m in messages]}\n\n"
        f"Напиши обновленное саммари. Верни только текст саммари."
    )
    response = llm.invoke([HumanMessage(content=summary_prompt)])
    messages_to_delete = messages[:-2]
    delete_commands = [RemoveMessage(id=m.id) for m in messages_to_delete]
    
    print(f"[SYSTEM] Удалено {len(delete_commands)} старых сообщений из памяти.")
    print(f"[SYSTEM] Новое саммари сохранено: {response.content}\n")
    
    return {"summary": response.content, "messages": delete_commands}

def route_after_chat(state: MemoryState) -> Literal["summarizer_node", "__end__"]:
    if len(state["messages"]) > 4:
        return "summarizer_node"
    return "__end__"

memory_builder = StateGraph(MemoryState)
memory_builder.add_node("chat_node", chat_node)
memory_builder.add_node("summarizer_node", summarizer_node)
memory_builder.add_edge(START, "chat_node")
memory_builder.add_conditional_edges("chat_node", route_after_chat)
memory_builder.add_edge("summarizer_node", END)

checkpointer = MemorySaver()
memory_app = memory_builder.compile(checkpointer=checkpointer)

def run_memory_agent():
    config = {"configurable": {"thread_id": "user_nlp_1"}}
    display(Markdown("`model`: Привет! Я NLP агент с памятью. (Введи 'exit' для выхода)"))
    
    while True:
        user_input = input("user : ")
        display(Markdown(f"`user`: {user_input}"))
        if user_input.strip().lower() == "exit":
            break
            
        for chunk in memory_app.stream({"messages": [HumanMessage(content=user_input)]}, config, stream_mode="updates"):
            if "chat_node" in chunk:
                display(Markdown(f"`model`: {chunk['chat_node']['messages'][0].content}"))

run_memory_agent()

`model`: Привет! Я NLP агент с памятью. (Введи 'exit' для выхода)

`user`: Мне 8 лет. Я люблю трансформеров, особенно Старскрима, и варить вкусный суп. Объясни мне что такое FastText

`model`: Привет! Здорово, что ты любишь трансформеров и варить суп! Теперь давай поговорим о FastText.

FastText — это специальный инструмент, который помогает компьютерам понимать текст. Представь, что у тебя есть много слов, и ты хочешь, чтобы компьютер знал, что некоторые слова похожи друг на друга. Например, "кот" и "кошечка" — это похожие слова, потому что они оба о кошках.

FastText делает это, разбивая слова на маленькие части, которые называются "н-граммами". Это как если бы ты взял слово "суп" и посмотрел на его части, например, "с", "у", "п". Так компьютер может лучше понять, как слова связаны друг с другом.

Кроме того, FastText может быстро находить похожие слова и помогать в задачах, таких как перевод текста или определение настроения. Это очень полезно, когда нужно работать с большим количеством текста!

Если у тебя есть еще вопросы, не стесняйся спрашивать!

`user`: А как он связан с Word2Vec?

`model`: Отличный вопрос! FastText и Word2Vec — это оба инструмента, которые помогают компьютерам понимать слова и их значения, но они работают немного по-разному.

Word2Vec создает векторы (это как набор чисел) для каждого слова, основываясь на том, как часто слова встречаются рядом друг с другом в тексте. Например, если слово "кошка" часто встречается рядом со словами "мяу" и "питомец", Word2Vec создаст вектор для "кошка", который будет похож на векторы для "мяу" и "питомец". Это помогает компьютеру понять, что эти слова связаны.

FastText, с другой стороны, улучшает идею Word2Vec, разбивая слова на маленькие части (н-граммы), как я уже говорил. Это значит, что даже если компьютер не видел слово раньше, он может понять его значение, глядя на его части. Например, если он знает, что "кошка" и "кошечка" похожи, он сможет понять и новое слово "кошачий", потому что оно содержит знакомую часть "кош".

Таким образом, FastText может работать лучше с новыми словами и редкими словами, чем Word2Vec. Оба метода полезны, но FastText дает больше возможностей для понимания языка!

Если у тебя есть еще вопросы, спрашивай!

...

`model`: Существует несколько других методов и инструментов, которые могут преобразовывать слова в числовые векторы, помимо FastText и Word2Vec. Вот некоторые из них:

1. **GloVe (Global Vectors for Word Representation)**: Это метод, который создает векторы слов на основе статистики словосочетаний в большом корпусе текста. GloVe использует матрицы соотношений слов и их частот, чтобы создать векторы, которые отражают семантические связи между словами.

2. **ELMo (Embeddings from Language Models)**: Этот метод создает контекстуальные векторы слов, которые зависят от контекста, в котором они используются. ELMo использует рекуррентные нейронные сети (RNN) для анализа текста и может учитывать, как одно и то же слово может иметь разные значения в разных предложениях.

3. **BERT (Bidirectional Encoder Representations from Transformers)**: Это более современный метод, который также создает контекстуальные векторы. BERT использует архитектуру трансформеров и обучается на задаче предсказания пропущенных слов в предложениях, что позволяет ему учитывать контекст с обеих сторон слова.

4. **Sentence Transformers**: Это расширение BERT, которое позволяет создавать векторы не только для отдельных слов, но и для целых предложений или текстов. Это полезно для задач, связанных с семантическим поиском и сопоставлением текстов.

Каждый из этих методов имеет свои особенности и может быть более подходящим для различных задач в обработке естественного языка. Если у тебя есть вопросы о каком-то из этих методов или ты хочешь узнать больше, дай знать!

`user`: Какая из них самая крутая?

`model`: Каждый из методов имеет свои сильные стороны и подходит для разных задач, поэтому "самая крутая" модель зависит от того, что именно ты хочешь сделать. Вот краткий обзор:

1. **GloVe**: Хорош для задач, где важна глобальная статистика слов, но не учитывает контекст. Подходит для простых задач, таких как классификация текста.

2. **Word2Vec**: Отлично работает для создания векторов слов и выявления семантических связей, но также не учитывает контекст. Хорош для задач, где важны отношения между словами.

3. **FastText**: Преимущество в том, что он может обрабатывать редкие и новые слова, разбивая их на н-граммы. Это делает его полезным для языков с богатой морфологией или для текстов с неформальным языком.

4. **ELMo**: Создает контекстуальные векторы, что делает его более мощным для задач, где значение слова зависит от контекста. Хорош для более сложных задач, таких как анализ тональности или извлечение информации.

5. **BERT**: Один из самых мощных методов на сегодняшний день. Он учитывает контекст с обеих сторон слова и показывает отличные результаты на многих задачах, включая вопросно-ответные системы и понимание текста. BERT и его производные (например, RoBERTa, DistilBERT) часто считаются "самыми крутыми" в современных NLP задачах.

6. **Sentence Transformers**: Если тебе нужно работать с предложениями или текстами, а не только со словами, это отличный выбор, так как он позволяет создавать векторы для целых предложений.

Если ты интересуешься конкретной задачей, я могу помочь выбрать наиболее подходящий метод!


[SYSTEM] Вызван узел суммаризации. Сжимаем контекст...
[SYSTEM] Удалено 4 старых сообщений из памяти.
[SYSTEM] Новое саммари сохранено: Пользователь, 8 лет, интересуется трансформерами, особенно Старскримом, и варкой супа. Он спрашивает о FastText, на что ему объясняют, что это инструмент для понимания текста, который разбивает слова на н-граммы, что помогает находить связи между словами. Пользователь интересуется, как FastText связан с Word2Vec. Ему объясняют, что оба инструмента создают векторы для слов, но FastText лучше справляется с новыми и редкими словами, разбивая их на части. Пользователь также спрашивает, знает ли FastText о Оптимусе Прайме. Ему объясняют, что FastText не "знает" о конкретных персонажах, но может анализировать текст, в котором они упоминаются, и находить связи с другими словами. Затем пользователь интересуется, какие еще методы могут преобразовывать слова в числовые векторы. Ему рассказывают о нескольких других методах, таких как GloVe, ELMo, BERT и Sentence

`user`: exit